In [2]:
%pip install prophet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 142.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 88.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [prophet]m3/4 [prophet]]
Note: you may need to restart the kernel to use updated packages.


In [5]:
# ── 1. Importação de Bibliotecas: Modelagem, Visualização e Tracking ──

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modelos
from sklearn.linear_model import LinearRegression # Baseline
from prophet import Prophet                         # Modelo Principal

# Métricas de Avaliação
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

# Rastreamento de Experimentos (MLflow)
import mlflow
import mlflow.sklearn
import mlflow.prophet

# Configuração do MLflow com o SageMaker Tracking Server
ARN_MLFLOW = "arn:aws:sagemaker:us-east-1:594715260047:mlflow-tracking-server/mlflow-ai-models-prod-ia"

# Aponta o MLflow para o servidor gerenciado da AWS
mlflow.set_tracking_uri(ARN_MLFLOW)

# Define o experimento no servidor remoto
mlflow.set_experiment("Proj_Cashback_RaiaDrogasil")

2026/10/09 19:06:20 INFO mlflow.tracking.fluent: Experiment with name 'Proj_Cashback_RaiaDrogasil' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://ai-models-prod-ia-mlflow-artifacts-594715260047/mlflow-tracking/3', creation_time=1791572780140, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1791572780140, lifecycle_stage='active', name='Proj_Cashback_RaiaDrogasil', tags={}, trace_location=None, workspace='default'>

In [ ]:
# ── Definições Estritas de Governança para o MLflow (ADR-0007) ──

# 1. Tags de Nível de Modelo (Registered Model Tags)
# Identificam a propriedade e o impacto do modelo no portfólio corporativo.
registered_model_tags = {
    "macro_area": "lealdade",
    "squad": "cross",
    "business_criticality": "tier_1", 
    "data_classification": "interno"
}

# 2. Tags de Nível de Versão 
# Rastreiam os metadados técnicos específicos da rodada de experimentação.
model_version_tags = {
    "algorithm": "prophet", # Altere para "linear_regression" ao rodar o baseline
    "training_dataset_ref": "model-proj-tpv-rd/data/fl_model_report.csv",
    "compliance_status": "pending",
    "sagemaker_instance_type": "ml.m5.large"
}

# 3. Aliases de Roteamento (Model Version Routing)
# Controle de transição de ciclo de vida que será utilizado no Model Registry
# (A API do MLflow exige a passagem sem o caractere '@' no método set_registered_model_alias)
alias_atual = "challenger" # Iniciamos a experimentação como challenger

aliases_esteira = {
    "producao": "champion",
    "validacao": "challenger",
    "homologacao": "staging",
    "seguranca": "fallback"
}